# BigAlpha2 depth imbalance stability submission draft

This candidate ranks stocks with more stable intraday top-3 bid/ask depth
imbalance higher. Local validation on e2e 5m passes 2024Q1-Q4 and full-year
2024; e2e 1m 2024Q1 also passes with low correlation to current submissions.


In [ ]:
def main(datasources, start_date, end_date):
    """
    Build a daily order-book imbalance stability factor for BigAlpha2.

    The factor is larger when top-3 bid/ask depth imbalance is more stable
    during the same trading day.
    """

    import numpy as np
    import pandas as pd
    import dai
    import structlog

    logger = structlog.get_logger()

    bar1m = datasources["bar1m"]
    query_start_date = pd.to_datetime(start_date) - pd.Timedelta(days=2)

    sql = f"""
    WITH cte_bar1m AS (
        SELECT
            date,
            instrument,
            (
                CASE WHEN bid_volume1 > 0 THEN bid_volume1 ELSE 0 END +
                CASE WHEN bid_volume2 > 0 THEN bid_volume2 ELSE 0 END +
                CASE WHEN bid_volume3 > 0 THEN bid_volume3 ELSE 0 END
            ) AS bid_depth,
            (
                CASE WHEN ask_volume1 > 0 THEN ask_volume1 ELSE 0 END +
                CASE WHEN ask_volume2 > 0 THEN ask_volume2 ELSE 0 END +
                CASE WHEN ask_volume3 > 0 THEN ask_volume3 ELSE 0 END
            ) AS ask_depth,
            strftime(date, '%Y-%m-%d') AS trading_day
        FROM {bar1m}
    ),
    cte_imbalance AS (
        SELECT
            trading_day,
            instrument,
            date,
            CASE
                WHEN bid_depth + ask_depth > 0
                THEN 1.0 * (bid_depth - ask_depth) / (bid_depth + ask_depth)
                ELSE NULL
            END AS imbalance
        FROM cte_bar1m
    ),
    cte_daily AS (
        SELECT
            trading_day,
            instrument,
            COUNT(imbalance) AS imbalance_count,
            STDDEV_POP(imbalance) AS imbalance_std
        FROM cte_imbalance
        GROUP BY trading_day, instrument
    )
    SELECT
        CAST(trading_day AS DATETIME) AS date,
        instrument,
        CASE
            WHEN imbalance_count >= 20 AND imbalance_std IS NOT NULL
            THEN -1.0 * imbalance_std
            ELSE NULL
        END AS factor
    FROM cte_daily
    ORDER BY date, instrument
    """

    logger.info("查询分钟盘口并构造深度不平衡稳定因子", start=str(start_date), end=str(end_date))
    df = dai.query(
        sql,
        filters={"date": [query_start_date.strftime("%Y-%m-%d %H:%M:%S"), end_date]},
        compression=True,
    ).df()

    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype(str)
    df["factor"] = pd.to_numeric(df["factor"], errors="coerce").replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["factor"])

    start_ts = pd.to_datetime(start_date).normalize()
    end_ts = pd.to_datetime(end_date).normalize()
    df = df[(df["date"] >= start_ts) & (df["date"] <= end_ts)]

    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"]).dt.normalize()
    stk_pool["instrument"] = stk_pool["instrument"].astype(str)

    result = pd.merge(
        df[["date", "instrument", "factor"]],
        stk_pool,
        how="inner",
        on=["date", "instrument"],
    )
    result = (
        result.drop_duplicates(["date", "instrument"], keep="last")
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
    )
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce").replace([np.inf, -np.inf], np.nan)
    result = result.dropna(subset=["factor"]).reset_index(drop=True)
    logger.info("深度不平衡稳定因子构建完成", rows=len(result))
    return result[["date", "instrument", "factor"]]


In [ ]:
if __name__ == "__main__":
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
        "financial": "bigalpha_2026_financial",
    }
    start_date = "2024-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"

    logger.info("计算深度不平衡稳定因子", start=start_date, end=end_date)
    factor_data = main(datasources, start_date, end_date)

    logger.info("读取官方因子库做本地回归评估", start=start_date, end=end_date)
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={"date": [start_date, end_date]},
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )
